In [1]:
import os
from sentence_transformers import SentenceTransformer
import requests

# print(os.environ["REQUESTS_CA_BUNDLE"])
print(os.environ["SSL_CERT_FILE"]) 
# print(os.environ["CURL_CA_BUNDLE"])


print("Testing HTTPS...")
resp = requests.get("https://google.com", timeout=20)
print("Status:", resp.status_code)

print("Loading model...")
model = SentenceTransformer(model_name_or_path = "sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded.")

c:\Users\yjaquier\.conda\envs\ollama\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\yjaquier\.conda\envs\ollama\Library\ssl\cacert.pem
Testing HTTPS...
Status: 200
Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2240.27it/s]


Model loaded.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]
embeddings = model.encode(sentences)
print(embeddings.shape)

similarities = model.similarity(embeddings, embeddings)
print(similarities)
print(embeddings[0])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2721.93it/s]


(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])
[ 1.91957522e-02  1.20085329e-01  1.59598291e-01  6.70659095e-02
  5.00748083e-02 -2.59187017e-02  5.64681962e-02 -9.28577334e-02
 -3.76114510e-02  6.32387493e-03 -4.28877473e-02  4.02834639e-03
  4.72777523e-03  3.24675962e-02  4.95197773e-02  5.29818274e-02
 -4.04454172e-02 -2.14836895e-02 -3.02760303e-02  2.20858455e-02
 -1.60775855e-01  8.08077976e-02 -2.80130561e-02  8.06255937e-02
 -2.85814330e-02  5.35818525e-02  1.26382308e-02  4.79190946e-02
  5.71111357e-03 -3.25831212e-02 -2.61571705e-02  8.00957531e-02
  1.47315878e-02 -3.24082486e-02 -4.12552394e-02 -9.68343113e-03
  5.00759401e-04 -1.56286821e-01 -6.77877367e-02  4.88779135e-02
  1.88976154e-02 -7.97722489e-02  2.43868306e-02  5.46689844e-03
  1.10656386e-02 -7.77949253e-03 -2.20388342e-02  3.50319929e-02
  1.06080189e-01 -4.72453795e-03 -5.78185171e-02  2.34565753e-02
 -5.92035577e-02 -6.09013177e-02 -7.7856823

In [ ]:
from pathlib import Path
from constants import SKIP_DIRS
from BM25 import chunks_from_markdown_files 
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading model...")
model = SentenceTransformer(model_name_or_path = "sentence-transformers/all-MiniLM-L6-v2")

chunks = chunks_from_markdown_files()
# for i, chunk in enumerate(chunks):
#   # print(f"[{i+1}] ({len(chunk["chunk"])} chunks):")
#   # print(chunk[:80] + "..." if len(chunk) > 80 else chunk)
#   print(chunks[i])

print("Generating embeddings...")
sentences = [chunk["chunk"] for chunk in chunks]
embeddings = model.encode(inputs = sentences, show_progress_bar = False, convert_to_numpy = True)
# print(embeddings.shape)

print("Computing question similarities...")
query = "How to generate an AWR report in HTML format ?"
query_embedding = model.encode(inputs = [query], show_progress_bar = False, convert_to_numpy = True)
scores = model.similarity(query_embedding, embeddings)
# print(scores)
# if hasattr(scores, "detach"):  # torch tensor
#   scores = scores.detach().cpu().numpy()

sorted_scores, sorted_indices = scores.flatten().sort(descending = True)
print(sorted_scores)
print(sorted_indices)
top_k = 10
for rank, idx in enumerate(sorted_indices[:top_k], start=1):
  print(f"--------------------------------------------------------------------------------\nRank {rank:02d} (score: {scores.flatten()[idx]:.4f}), File: {chunks[idx]['file']}\nChunk: {chunks[idx]['chunk']}")


resource module not available on Windows


Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2717.10it/s]


Generating embeddings...
Computing question similarities...
tensor([ 0.6965,  0.6813,  0.6512,  ..., -0.1596, -0.1744, -0.1816])
tensor([3190, 3192, 4062,  ...,  972, 2289, 1525])
--------------------------------------------------------------------------------
Rank 01 (score: 0.6965), File: skills-main\db\performance\awr-reports.md
Chunk: ## generating awr reports
--------------------------------------------------------------------------------
Rank 02 (score: 0.6813), File: skills-main\db\performance\awr-reports.md
Chunk: ### html report (preferred for readability)

```sql
select output
from   table(
         dbms_workload_repository.awr_report_html(
           l_dbid      => (select dbid from v$database),
           l_inst_num  => 1,
           l_bid       => 200,
           l_eid       => 201
         )
       );
```
--------------------------------------------------------------------------------
Rank 03 (score: 0.6512), File: skills-main\db\sqlcl\sqlcl-awr.md
Chunk: ### awr create r

In [4]:
sorted_scores, sorted_indices = scores.flatten().sort(descending=True)

print(sorted_scores[:top_k])
print(sorted_indices[:top_k])

print(len(chunks))
print(len(sorted_scores))

idx = 3190
print(
  f"--------------------------------------------------------------------------------\n"
  f"Index {idx} (score: {scores.flatten()[idx]:.4f}), "
  f"File: {chunks[idx]['file']}\n"
  f"Chunk: {chunks[idx]['chunk']}"
)

tensor([0.6965, 0.6813, 0.6512, 0.6366, 0.5914, 0.5882, 0.5694, 0.5586, 0.5560,
        0.5251])
tensor([3190, 3192, 4062, 4064, 4067, 4066, 4065, 2762, 3179, 4429])
4436
4436
--------------------------------------------------------------------------------
Index 3190 (score: 0.6965), File: skills-main\db\performance\awr-reports.md
Chunk: ## generating awr reports


In [14]:
scores = model.similarity(query_embedding, embeddings)
unsorted_with_doc_ids = [
  {"id": doc_id, "file": chunks[doc_id]["file"], "chunk": chunks[doc_id]["chunk"], "embedding_score": float(scores.flatten()[doc_id])}
  for doc_id in range(len(chunks))
]

# optional preview
print(unsorted_with_doc_ids[:10])
# print(scores)
# unsorted_scores, unsorted_indices = scores.flatten().sort(descending=False)
# print(unsorted_scores)
# print(unsorted_indices)

[{'id': 0, 'file': 'skills-main\\db\\admin\\backup-recovery.md', 'chunk': '# oracle rman backup and recovery', 'embedding_score': 0.11422142386436462}, {'id': 1, 'file': 'skills-main\\db\\admin\\backup-recovery.md', 'chunk': "## overview\n\nrecovery manager (rman) is oracle's primary tool for database backup, restore, and recovery operations. it provides block-level backup integrity checking, compression, encryption, incremental backups, and tight integration with the oracle database engine. rman eliminates the need for manual backup scripting and ensures consistent backups even for an open database.\n\nunderstanding backup and recovery is the single most critical skill for any oracle dba. a database that cannot be recovered is a database that cannot be trusted.\n\n---", 'embedding_score': 0.08638248592615128}, {'id': 2, 'file': 'skills-main\\db\\admin\\backup-recovery.md', 'chunk': '## rman architecture', 'embedding_score': 0.15260881185531616}, {'id': 3, 'file': 'skills-main\\db\\adm

In [ ]:
import importlib
import BM25
importlib.reload(BM25)
import embeddings
importlib.reload(embeddings)
from BM25 import chunks_from_markdown_files
from embeddings import embedding_score

chunks = chunks_from_markdown_files()
query = "How to generate an AWR report in HTML format ?"
embedding_score_dict = embedding_score(query, chunks)
for item in sorted(embedding_score_dict, key=lambda x: x["embedding_score"], reverse=True)[:10]:
  print(item)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2693.33it/s]


{'id': 2553, 'source': 'skills-main\\db\\performance\\awr-reports.md', 'section_id': 9, 'chunk_id': 0, 'header_path': ['AWR Reports — Automatic Workload Repository', 'Generating AWR Reports', 'HTML Report (preferred for readability)'], 'chunk': '```sql\nSELECT output\nFROM   TABLE(\nDBMS_WORKLOAD_REPOSITORY.AWR_REPORT_HTML(\nl_dbid      => (SELECT dbid FROM v$database),\nl_inst_num  => 1,\nl_bid       => 200,\nl_eid       => 201\n)\n);\n```', 'embedding_score': 0.7179502844810486}
{'id': 2233, 'source': 'skills-main\\db\\monitoring\\top-sql-queries.md', 'section_id': 22, 'chunk_id': 0, 'header_path': ['Top SQL Queries and SQL Monitoring', 'Combining V$ and AWR Data: A Practical Workflow', 'Step 4: Generate SQL Monitoring Report'], 'chunk': "```sql\n-- Get the latest sql_exec_id for this SQL\nSELECT sql_exec_id, sql_exec_start, elapsed_time/1e6 AS elapsed_sec, status\nFROM   v$sql_monitor\nWHERE  sql_id = 'abc123def456g'\nORDER BY sql_exec_start DESC\nFETCH FIRST 1 ROW ONLY;\n```  \nThe